# Learning Theory — Preliminaries

## Learning Objectives
- State and apply the **Union Bound** to bound the probability of any one of $k$ events
- State and simulate **Hoeffding's Inequality** (the Chernoff bound for Bernoulli random variables)
- Define **training error** $\hat{\varepsilon}(h)$ and **generalisation error** $\varepsilon(h)$
- Define the **hypothesis class** $\mathcal{H}$ and understand Empirical Risk Minimisation (ERM)
- Understand the PAC learning framework and the role of i.i.d. assumption

## Problem Statement

We are given a training set $S = \{(x^{(i)}, y^{(i)}); i = 1, \ldots, m\}$ drawn i.i.d. from a distribution $\mathcal{D}$.

For binary classification with $y \in \{0, 1\}$ we define:

$$\hat{\varepsilon}(h) = \frac{1}{m}\sum_{i=1}^{m} \mathbf{1}\{h(x^{(i)}) \neq y^{(i)}\} \qquad \text{(training / empirical error)}$$

$$\varepsilon(h) = P_{(x,y) \sim \mathcal{D}}(h(x) \neq y) \qquad \text{(generalisation error)}$$

The learning algorithm picks:
$$\hat{h} = \arg\min_{h \in \mathcal{H}} \hat{\varepsilon}(h) \qquad \text{(Empirical Risk Minimisation)}$$

The central question of learning theory: **can we bound $\varepsilon(\hat{h})$?**

## 1. The Union Bound

**Lemma (Union Bound).** For any $k$ events $A_1, \ldots, A_k$ (not necessarily independent):

$$P(A_1 \cup \cdots \cup A_k) \leq P(A_1) + \cdots + P(A_k)$$

This is the simplest tool for controlling "at least one bad event happens."  
If each individual event has probability at most $p$, then the probability that **any** one occurs is at most $k \cdot p$.

In learning theory we apply this to bound:
$$P\bigl(\exists\, h_i \in \mathcal{H} : |\varepsilon(h_i) - \hat{\varepsilon}(h_i)| > \gamma\bigr) \leq \sum_{i=1}^{k} P\bigl(|\varepsilon(h_i) - \hat{\varepsilon}(h_i)| > \gamma\bigr)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Simulate: k events each with P(A_i) = p; what is P(any A_i occurs)?
k_values = [1, 2, 5, 10, 20, 50]
p_each   = 0.05
n_trials = 100_000

empirical_union = []
union_bound     = []

for k in k_values:
    # Simulate k independent events, each fires with probability p_each
    events    = np.random.rand(n_trials, k) < p_each     # (n_trials, k) bool
    any_event = events.any(axis=1)                        # (n_trials,)
    empirical_union.append(any_event.mean())
    union_bound.append(min(k * p_each, 1.0))

# True value for independent events: P(none fires) = (1-p)^k
true_union = [1 - (1 - p_each) ** k for k in k_values]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, union_bound,     'r-s', lw=2.5, ms=7, label=f'Union Bound: $k \\times {p_each}$')
ax.plot(k_values, empirical_union, 'b-o', lw=2.5, ms=7, label='Simulated $P$(any $A_i$)')
ax.plot(k_values, true_union,      'g--', lw=2,   label='True $P$ (independent events)')
ax.set_xlabel('Number of events $k$'); ax.set_ylabel('Probability')
ax.set_title(f'Union Bound vs. True Probability\n($P(A_i) = {p_each}$ each, {n_trials:,} trials)')
ax.legend(fontsize=10)
ax.text(0.03, 0.97,
        'Union bound is an upper bound\n— tightest when events are disjoint',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))
plt.tight_layout()
plt.show()

## 2. Hoeffding's Inequality (Chernoff Bound)

**Lemma (Hoeffding's Inequality).** Let $Z_1, \ldots, Z_m$ be i.i.d. $\text{Bernoulli}(\phi)$ random variables.  
Let $\hat{\phi} = \frac{1}{m}\sum_{i=1}^{m} Z_i$. Then for any $\gamma > 0$:

$$P(|\hat{\phi} - \phi| > \gamma) \leq 2 \exp(-2\gamma^2 m)$$

**Interpretation:** the sample mean $\hat{\phi}$ concentrates around the true mean $\phi$ exponentially fast as $m$ grows.  
Think of $\phi$ as the true error rate of a hypothesis and $\hat{\phi}$ as its measured training error — Hoeffding tells us training error is a reliable estimator of the true error, provided $m$ is large enough.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

phi   = 0.3    # true Bernoulli parameter
gamma = 0.1    # deviation threshold

m_values = np.logspace(1, 4, 60).astype(int)
n_trials = 5000

empirical_prob = []
hoeffding_bound = []

for m in m_values:
    # Draw n_trials estimates of phi, each from m coin flips
    samples = np.random.binomial(m, phi, size=n_trials) / m
    empirical_prob.append(np.mean(np.abs(samples - phi) > gamma))
    hoeffding_bound.append(2 * np.exp(-2 * gamma ** 2 * m))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: probability vs m
ax = axes[0]
ax.semilogy(m_values, empirical_prob,  'b-',  lw=2.5, label=r'Empirical $P(|\hat{\phi}-\phi|>\gamma)$')
ax.semilogy(m_values, hoeffding_bound, 'r--', lw=2.5, label=r'Hoeffding bound $2e^{-2\gamma^2 m}$')
ax.set_xlabel('Sample size $m$'); ax.set_ylabel('Probability (log scale)')
ax.set_title(f'Hoeffding Inequality ($\\phi={phi}$, $\\gamma={gamma}$)')
ax.legend(fontsize=9)
ax.text(0.55, 0.95,
        'Bound is loose but always valid\nDecay is exponential in $m$',
        transform=ax.transAxes, fontsize=8.5, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# Right: distribution of phi_hat for small vs large m
ax = axes[1]
for m_demo, color, label in [(20, '#d6604d', '$m=20$'), (200, '#2166ac', '$m=200$')]:
    samples = np.random.binomial(m_demo, phi, size=10000) / m_demo
    ax.hist(samples, bins=40, density=True, alpha=0.55, color=color, label=label)

ax.axvline(phi, color='k', lw=2, ls='--', label=f'True $\\phi={phi}$')
ax.axvspan(phi - gamma, phi + gamma, alpha=0.10, color='green',
           label=f'$|\\hat{{\\phi}}-\\phi|\\leq\\gamma={gamma}$')
ax.set_xlabel('$\\hat{\\phi}$'); ax.set_ylabel('Density')
ax.set_title('Distribution of $\\hat{\\phi}$ concentrates around $\\phi$ as $m$ grows')
ax.legend(fontsize=9)

fig.suptitle('Hoeffding Inequality — Empirical Verification', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Training Error, Generalisation Error, and ERM

### Training (Empirical) Error
$$\hat{\varepsilon}(h) = \frac{1}{m}\sum_{i=1}^{m} \mathbf{1}\{h(x^{(i)}) \neq y^{(i)}\}$$

This is the fraction of training examples misclassified by $h$. It is the quantity the learning algorithm directly minimises.

### Generalisation Error
$$\varepsilon(h) = P_{(x,y) \sim \mathcal{D}}(h(x) \neq y)$$

This is what we actually care about. We cannot compute it exactly but can estimate it via a held-out test set.

### Empirical Risk Minimisation (ERM)
ERM picks the hypothesis that minimises training error:
$$\hat{h} = \arg\min_{h \in \mathcal{H}} \hat{\varepsilon}(h)$$

For linear classification this means finding $\theta$ that minimises the training misclassification rate. Logistic regression can be viewed as a soft approximation to ERM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(1)

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                            n_informative=2, random_state=1,
                            n_clusters_per_class=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

clf = LogisticRegression()
clf.fit(X_tr, y_tr)

train_err = 1 - clf.score(X_tr, y_tr)
test_err  = 1 - clf.score(X_te, y_te)

xmin, xmax = X[:,0].min()-0.5, X[:,0].max()+0.5
ymin, ymax = X[:,1].min()-0.5, X[:,1].max()+0.5
xx, yy = np.meshgrid(np.linspace(xmin,xmax,300), np.linspace(ymin,ymax,300))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.15, cmap='RdBu')
ax.contour(xx,  yy, Z, levels=[0.5], colors='k', linewidths=2)

ax.scatter(X_tr[y_tr==1,0], X_tr[y_tr==1,1], marker='x', c='#2166ac',
           s=70, lw=2, label='Train $y=1$')
ax.scatter(X_tr[y_tr==0,0], X_tr[y_tr==0,1], marker='o',
           s=60, facecolors='none', edgecolors='#d6604d', lw=2, label='Train $y=0$')
ax.scatter(X_te[y_te==1,0], X_te[y_te==1,1], marker='x', c='#2166ac',
           s=70, lw=2, alpha=0.35, label='Test $y=1$')
ax.scatter(X_te[y_te==0,0], X_te[y_te==0,1], marker='o',
           s=60, facecolors='none', edgecolors='#d6604d', lw=2, alpha=0.35, label='Test $y=0$')

ax.set_title(f'ERM via Logistic Regression\n'
             f'Train error $\\hat{{\\varepsilon}}(\\hat{{h}}) = {train_err:.3f}$  |  '
             f'Test (gen.) error $\\varepsilon(\\hat{{h}}) \\approx {test_err:.3f}$')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=9)
ax.text(0.03, 0.03,
        'ERM minimises training error; we care about generalisation error.\n'
        'Their gap is bounded by learning theory (see next notebook).',
        transform=ax.transAxes, fontsize=8.5,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))
plt.tight_layout()
plt.show()

## 4. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="464"
     viewBox="0 0 780 464" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: i.i.d. training set -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">i.i.d. training set</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">S from &#x1d49f;</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >m examples drawn independently &#x2014; i.i.d. assumption is essential</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >ERM: &#x125; = argmin &#x3b5;&#x302;(h) over &#x1d4d7;</text>

  <!-- Row 2: ERM hypothesis -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">ERM hypothesis &#x125;</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >minimises &#x3b5;&#x302;(h) on S; we want to bound &#x3b5;(&#x125;) &#x2014; the true generalisation error</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="178" font-size="11.5" font-style="italic" fill="#555"
        >apply Hoeffding per hypothesis:</text>
  <text x="114" y="196" font-size="11.5" font-style="italic" fill="#555"
        >P(|&#x3b5;(h)&#x2212;&#x3b5;&#x302;(h)|&gt;&#x3b3;) &#x2264; 2e&#x207b;&#xb2;&#x3b3;&#xb2;m</text>

  <!-- Row 3: Hoeffding per hypothesis -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Hoeffding&#x2019;s</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">inequality</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >bounds the deviation of &#x3b5;&#x302;(h) from &#x3b5;(h) for a single fixed hypothesis</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >union bound over all k hypotheses in &#x1d4d7;</text>

  <!-- Row 4: Union bound -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Union bound</text>
  <text x="102" y="352" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">over &#x1d4d7;</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >P(&#x2203;h &#x2208; &#x1d4d7; : |&#x3b5;(h)&#x2212;&#x3b5;&#x302;(h)|&gt;&#x3b3;) &#x2264; 2|&#x1d4d7;| &#xb7; e&#x207b;&#xb2;&#x3b3;&#xb2;m &#x2014; controls the worst case</text>

  <!-- step 4&#x2192;5 -->
  <line x1="102" y1="358" x2="102" y2="408"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 5: Generalisation bound -->
  <rect x="10" y="412" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="435" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Generalisation</text>
  <text x="102" y="452" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">bound for &#x125;</text>
  <line x1="197" y1="435" x2="216" y2="435"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="412" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >&#x3b5;(&#x125;) &#x2264; &#x3b5;&#x302;(&#x125;) + &#x221a;(log(2|&#x1d4d7;|/&#x3b4;) / 2m) with probability &#x2265; 1 &#x2212; &#x3b4;</text>
</svg>
""")

## Summary

| Concept | Formula / Statement | Key Insight |
|---|---|---|
| Training error | $\hat{\varepsilon}(h) = \frac{1}{m}\sum_i \mathbf{1}\{h(x^{(i)}) \neq y^{(i)}\}$ | Quantity the algorithm minimises directly |
| Generalisation error | $\varepsilon(h) = P_{(x,y)\sim\mathcal{D}}(h(x) \neq y)$ | True measure of performance on unseen data |
| ERM | $\hat{h} = \arg\min_{h \in \mathcal{H}} \hat{\varepsilon}(h)$ | Learning reduces to minimising empirical error |
| Union Bound | $P(\bigcup_i A_i) \leq \sum_i P(A_i)$ | Combines per-hypothesis bounds across all of $\mathcal{H}$ |
| Hoeffding's Inequality | $P(\lvert\hat{\phi} - \phi\rvert > \gamma) \leq 2e^{-2\gamma^2 m}$ | Sample mean concentrates exponentially fast in $m$ |
| Generalisation bound (finite $\mathcal{H}$) | $\varepsilon(\hat{h}) \leq \hat{\varepsilon}(\hat{h}) + \sqrt{\frac{\log(2\lvert\mathcal{H}\rvert/\delta)}{2m}}$ | Holds with probability $\geq 1 - \delta$ |

**Key insight:** with a finite hypothesis class, training error is a reliable proxy for generalisation error when $m$ is large; the gap shrinks as $O(1/\sqrt{m})$ and grows only logarithmically with $|\mathcal{H}|$.